# Quantum Support Vector Machines in 4 Leptons Data Set - Atlas Open Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import uproot
import matplotlib.text
import sys
import awkward as ak
from atlasopenmagic import install_from_environment
import atlasopenmagic as atom
import qiskit
from qiskit_machine_learning.datasets import ad_hoc_data
from qiskit import QuantumCircuit, transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import minmax_scale
from sklearn.decomposition import PCA
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
atom.set_release('2025e-13tev-beta')

Fetching metadata for release: 2025e-13tev-beta...
Fetching datasets: 100%|██████████| 374/374 [00:00<00:00, 522.10datasets/s]
✓ Successfully cached 374 datasets.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


In [3]:
atom.available_skims()

['1LMET30',
 '2J2LMET30',
 '2bjets',
 '2muons',
 '2to4lep',
 '3J1LMET30',
 '3lep',
 '4lep',
 'GamGam',
 'exactly3lep',
 'exactly4lep']

In [4]:
skim='GamGam'
signal_list = atom.get_urls('346189', skim=skim, protocol= 'https', cache = True) #This is our pp->ttbarH->gamgam
bkg_GGFH = atom.get_urls('345944',skim=skim, protocol= 'https', cache = True)  #GGF H->gamma gamma also produces a higgs so it is similar but not the same
bkg_qqZH = atom.get_urls('341456',skim=skim, protocol= 'https', cache = True) #qq->ZH, H->WW, Z->veve
background_lists =  bkg_GGFH +bkg_qqZH          

In [5]:
print('signal:', signal_list) #29592 events
print('background:', background_lists)

signal: ['simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_GamGam_mc_346189.aMcAtNloPythia8EvtGen_ttH_gamgam.GamGam.root']
background: ['simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_GamGam_mc_345944.aMcAtNloPy8EG_A14NNPDF23LO_ppx0yy_FxFx_Np012_SM.GamGam.root', 'simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_GamGam_mc_341456.PowhegPythia8EvtGen_CT10_AZNLO_ZH125J_MINLO_veveWWlvqq_VpT.GamGam.root']


Going by Wu, et al. the features tha I'll be using will be the following

| Name of Feature   | Description |
| -------- | ------- |
| photon_pt  | The transverse momentum of the photons. <br> Since we analizing events with only two photons, <br> there are only 2 elements to this feature. <br> It is important to distinguish the photons by sorting <br> them according to their transverse momentum. <br> As such calling the photon with the highest transverse <br> momentum leading photon and the other one subleading   |
| photon_eta | pseudorapidity of our photons     |
| jet_pt    | Transverse momentum of our jets, here we can expect at most 6 jets <br> it is important to clarify that we may also end up <br> in an event with few or 0 jets |
| jet_n | pseudorapidity of our jets    |
| jet_btag_quantile |    did the jet at which we are looking to come from bottom quarks  |
| met    | missing transverse momentum, that is the energy that is missing from what we <br> would have awaited to obtain    |




In [7]:
data = []
for file in background_lists + signal_list:
    print(file)
    analysis = uproot.open(file + ':analysis')
    attributes = ['photon_pt', 'photon_eta', 'jet_pt', 'jet_eta','jet_btag_quantile', 'met']
    aliases_dict = {'photon_pt': 'photon_pt', 'photon_eta': 'photon_eta', 'jet_pt': 'jet_pt', ' jet_eta': 'jet_eta',
                    'btag': 'jet_btag_quantile', 'met': 'met'}
    partial_data = analysis.arrays(attributes , aliases = aliases_dict) ## casi me olvido del .arrays
    data=np.concatenate([data, partial_data] , axis = 0)
print(data)

simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_GamGam_mc_345944.aMcAtNloPy8EG_A14NNPDF23LO_ppx0yy_FxFx_Np012_SM.GamGam.root


/home/marti/cern_projects/.venv/lib/python3.12/site-packages/uproot/language/python.py:467: NameConflictWarning: 'photon_pt' is both an alias and a branch name
  warnings.warn(
/home/marti/cern_projects/.venv/lib/python3.12/site-packages/uproot/language/python.py:467: NameConflictWarning: 'photon_eta' is both an alias and a branch name
  warnings.warn(
/home/marti/cern_projects/.venv/lib/python3.12/site-packages/uproot/language/python.py:467: NameConflictWarning: 'jet_pt' is both an alias and a branch name
  warnings.warn(
/home/marti/cern_projects/.venv/lib/python3.12/site-packages/uproot/language/python.py:467: NameConflictWarning: 'met' is both an alias and a branch name
  warnings.warn(


simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_GamGam_mc_341456.PowhegPythia8EvtGen_CT10_AZNLO_ZH125J_MINLO_veveWWlvqq_VpT.GamGam.root
simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_GamGam_mc_346189.aMcAtNloPythia8EvtGen_ttH_gamgam.GamGam.root
[{photon_pt: [40, 34.8], photon_eta: [-2.22, ...], jet_pt: [], ...}, ..., {...}]


In [8]:
data.fields

['photon_pt', 'photon_eta', 'jet_pt', 'jet_eta', 'jet_btag_quantile', 'met']

In [9]:
labels = np.concatenate([np.zeros(123573-29592),np.ones(29592)]) #0 for background, 1 for signal
data_with_labels = ak.with_field(data, labels, 'labels' ) 

In [10]:
ak.to_numpy(ak.fill_none(ak.pad_none(data_with_labels.photon_pt, 5,axis = 1, clip=True),0, axis = 1)).shape

(123573, 5)

In [11]:
ak.to_numpy(data_with_labels.met).reshape(-1,1)

array([[18.807287],
       [12.411184],
       [17.644152],
       ...,
       [32.09612 ],
       [61.331306],
       [30.231699]], shape=(123573, 1), dtype=float32)

In [12]:
data_with_labels.met.show()
ak.to_numpy(data_with_labels.met).reshape(-1,1).shape

[18.8,
 12.4,
 17.6,
 48.1,
 33.1,
 39.1,
 21.4,
 29.9,
 13.9,
 7.93,
 ...,
 24.4,
 70,
 69.7,
 71,
 31.4,
 22.6,
 32.1,
 61.3,
 30.2]


(123573, 1)

In [13]:
ak.max(ak.num(data_with_labels.jet_btag_quantile,axis= 1))

np.int64(14)

In [14]:
sorter = np.argsort(data_with_labels.photon_pt, axis = 1)[:,::-1]
#TBD

In [15]:
vecs=np.array([], dtype=np.int64).reshape(123573,0)
print(vecs.shape)
sum=0
for f in data_with_labels.fields:
    if data_with_labels[f].type == data_with_labels.photon_pt.type or data_with_labels[f].type == data_with_labels.jet_btag_quantile.type:
        print(f ,':', ak.max(ak.num(data_with_labels[f],axis= 1)))
        max_n = ak.max(ak.num(data_with_labels[f],axis= 1)).item()
        s = ak.fill_none(ak.pad_none(data_with_labels[f], target=max_n,axis = 1, clip=True),0, axis = 1)
        sum = sum+ak.max(ak.num(data_with_labels[f],axis= 1))
        vecs = np.hstack((vecs, ak.to_numpy(s)))
    else:
        print(f ,':', 1)
        vecs = np.hstack((vecs, ak.to_numpy(data_with_labels[f]).reshape(-1,1)))
        sum = sum+1
print('sum:', sum)
print('vecs shape:', vecs.shape)

(123573, 0)
photon_pt : 5
photon_eta : 5
jet_pt : 14
jet_eta : 14
jet_btag_quantile : 14
met : 1
labels : 1
sum: 54
vecs shape: (123573, 54)


In [16]:
vecs.shape

(123573, 54)

In [84]:
pca = PCA(n_components=7)
decomposed = pca.fit_transform(vecs[:,:-1])
decomposed

array([[-7.95449618e+01, -1.62118520e+01,  2.49353678e+00, ...,
        -6.61796719e+00, -8.35260970e-01,  1.35222083e-01],
       [-4.29082695e+01, -5.84909303e+00, -1.61775597e+01, ...,
        -7.74633698e+00, -3.41258687e+00,  2.33642680e-01],
       [-6.79990124e+01,  3.97595037e+00, -2.82649542e+00, ...,
         1.85146038e+01, -1.02945883e+00,  5.06932084e-02],
       ...,
       [ 1.22525744e+01, -4.81968785e+01,  7.32812843e+00, ...,
        -7.07678436e-01, -3.30726973e+00,  3.50315012e+00],
       [ 1.54313062e+02,  2.20297362e+00,  1.18136031e+01, ...,
         5.23945205e+01, -2.72245347e+01,  2.23507143e+00],
       [ 8.29436480e+01,  9.53879288e+00, -4.26646812e+00, ...,
        -3.46277668e+01, -2.51184944e+01, -1.30861803e+01]],
      shape=(123573, 7))

53 features is kind of a lot. Speacially for something as slow as a quantum computer; as such we shall use PCA to extract the "most important" infromation out of our dataset

In [95]:
vecs_scaled = minmax_scale(decomposed,(-np.pi,np.pi))
vecs_scaled=np.hstack((vecs_scaled,vecs[:,-1].reshape(-1,1)))
np.random.shuffle(vecs_scaled) #reduce the number of events to 100 for testing purposes
vecs_reduced = vecs_scaled[:100,:]
vecs_reduced.shape

(100, 8)

In [96]:
x = vecs_reduced[:,:-1]
y = vecs_reduced[:,-1]
X_train, X_test,Y_train,Y_test = train_test_split (
    x,y,test_size=0.2, random_state=42,shuffle=True
)

In [97]:
backend = BasicSimulator()
backend.set_options(
    shots = 1024,
    seed_simulator = 42
)

In [98]:
feature_map = zz_feature_map(
    feature_dimension=7,
    reps= 5,
    entanglement= "circular"
)


In [99]:
kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

In [100]:
qsvc_test = QSVC(quantum_kernel=kernel)
qsvc_test.fit(X_train,Y_train)
qsvc_test.score(X_test,Y_test)

0.8

Surprisingly acurrate